# مولد فيديوهات الكورسات من النصوص
هذا Notebook يقوم بتحويل نصوص الكورسات إلى فيديوهات تعليمية باستخدام:
1. قراءة ملفات النصوص من مجلد video-scripts
2. تحويل النص العربي إلى صوت باستخدام TTS
3. إنشاء شرائح فيديو من النص
4. دمج الصوت والفيديو في ملف نهائي

In [ ]:
# Install required packages
!pip install gtts edge-tts moviepy pillow arabic-reshaper reportlab
!pip install opencv-python-headless

In [ ]:
import os
import re
from pathlib import Path
from gtts import gTTS
from moviepy.editor import *
from PIL import Image, ImageDraw, ImageFont
import arabic_reshaper
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
import json

# Configuration
SCRIPTS_DIR = Path('video-scripts')
OUTPUT_DIR = Path('videos')
AUDIO_DIR = OUTPUT_DIR / 'audio'
SLIDES_DIR = OUTPUT_DIR / 'slides'
FINAL_DIR = OUTPUT_DIR / 'final'

# Create directories
for dir_path in [OUTPUT_DIR, AUDIO_DIR, SLIDES_DIR, FINAL_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print("Directories created successfully")

In [ ]:
def parse_script_file(file_path):
    """Parse markdown script file and extract sections"""
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Extract title (first heading)
    title_match = re.search(r'^#\s+(.+)$', content, re.MULTILINE)
    title = title_match.group(1) if title_match else file_path.stem
    
    # Extract sections (## headings)
    sections = re.split(r'^##\s+', content, flags=re.MULTILINE)[1:]
    
    parsed_sections = []
    for i, section in enumerate(sections):
        lines = section.strip().split('\n')
        section_title = lines[0] if lines else f'Section {i+1}'
        section_content = '\n'.join(lines[1:]) if len(lines) > 1 else ''
        parsed_sections.append({
            'title': section_title,
            'content': section_content
        })
    
    return {
        'title': title,
        'sections': parsed_sections
    }

In [ ]:
def text_to_speech_arabic(text, output_path, lang='ar'):
    """Convert Arabic text to speech using gTTS"""
    try:
        tts = gTTS(text=text, lang=lang, slow=False)
        tts.save(str(output_path))
        print(f"Audio saved to: {output_path}")
        return True
    except Exception as e:
        print(f"Error generating TTS: {e}")
        return False

In [ ]:
def create_slide(title, content, output_path, width=1920, height=1080):
    """Create a slide image with Arabic text"""
    # Create image with dark background
    img = Image.new('RGB', (width, height), color='#0A1628')
    draw = ImageDraw.Draw(img)
    
    # Try to use Arabic font, fallback to default
    try:
        font_large = ImageFont.truetype('arial.ttf', 60)
        font_medium = ImageFont.truetype('arial.ttf', 40)
    except:
        font_large = ImageFont.load_default()
        font_medium = ImageFont.load_default()
    
    # Reshape Arabic text for proper rendering
    title_reshaped = arabic_reshaper.reshape(title)
    content_reshaped = arabic_reshaper.reshape(content)
    
    # Draw title
    draw.text((100, 100), title_reshaped, fill='#00D4AA', font=font_large)
    
    # Draw content (wrapped)
    y_position = 300
    max_width = width - 200
    
    # Simple text wrapping
    words = content_reshaped.split()
    line = ''
    for word in words:
        test_line = line + ' ' + word if line else word
        bbox = draw.textbbox((0, 0), test_line, font=font_medium)
        if bbox[2] < max_width:
            line = test_line
        else:
            if line:
                draw.text((100, y_position), line, fill='#F0F4FF', font=font_medium)
                y_position += 60
            line = word
    
    if line:
        draw.text((100, y_position), line, fill='#F0F4FF', font=font_medium)
    
    # Add decorative elements
    draw.rectangle((50, 50), (width-50, height-50), outline='#00D4AA', width=3)
    
    img.save(output_path)
    print(f"Slide saved to: {output_path}")
    return output_path

In [ ]:
def generate_video_from_script(script_file):
    """Generate video from a single script file"""
    script_path = Path(script_file)
    if not script_path.exists():
        print(f"Script file not found: {script_path}")
        return None
    
    # Parse script
    script_data = parse_script_file(script_path)
    print(f"Processing: {script_data['title']}")
    print(f"Sections: {len(script_data['sections'])}")
    
    # Create output subdirectory for this video
    video_name = script_path.stem
    video_audio_dir = AUDIO_DIR / video_name
    video_slides_dir = SLIDES_DIR / video_name
    
    for dir_path in [video_audio_dir, video_slides_dir]:
        dir_path.mkdir(parents=True, exist_ok=True)
    
    audio_clips = []
    slide_images = []
    
    # Process each section
    for idx, section in enumerate(script_data['sections']):
        print(f"\nProcessing section {idx+1}: {section['title']}")
        
        # Combine title and content for narration
        narration_text = f"{section['title']}. {section['content']}"
        
        # Generate audio
        audio_path = video_audio_dir / f"section_{idx+1}.mp3"
        if text_to_speech_arabic(narration_text, audio_path):
            audio_clips.append(audio_path)
        
        # Create slide
        slide_path = video_slides_dir / f"slide_{idx+1}.png"
        slide_content = section['content'][:200] + '...' if len(section['content']) > 200 else section['content']
        create_slide(section['title'], slide_content, slide_path)
        slide_images.append(slide_path)
    
    # Create video from slides and audio
    if audio_clips and slide_images:
        clips = []
        
        for audio_path, slide_path in zip(audio_clips, slide_images):
            # Load audio
            audio = AudioFileClip(str(audio_path))
            
            # Load slide image
            slide = ImageClip(str(slide_path)).set_duration(audio.duration)
            
            # Set audio to slide
            video_clip = slide.set_audio(audio)
            clips.append(video_clip)
        
        # Concatenate all clips
        final_video = concatenate_videoclips(clips)
        
        # Save final video
        output_path = FINAL_DIR / f"{video_name}.mp4"
        final_video.write_videofile(
            str(output_path),
            fps=24,
            codec='libx264',
            audio_codec='aac'
        )
        
        print(f"\n✅ Video saved to: {output_path}")
        return output_path
    else:
        print("No audio or slides generated")
        return None

In [ ]:
# List all script files
script_files = list(SCRIPTS_DIR.glob('*.md'))
print(f"Found {len(script_files)} script files:")
for file in script_files:
    print(f"  - {file.name}")

In [ ]:
# Generate videos for all scripts
generated_videos = []

for script_file in script_files:
    print(f"\n{'='*60}")
    video_path = generate_video_from_script(script_file)
    if video_path:
        generated_videos.append(video_path)

print(f"\n{'='*60}")
print(f"\n✅ Generated {len(generated_videos)} videos successfully!")
print("\nGenerated videos:")
for video in generated_videos:
    print(f"  - {video}")

In [ ]:
# Alternative: Generate video for a specific script
# specific_script = 'video-scripts/capa-1-1.md'
# generate_video_from_script(specific_script)